<a href="https://colab.research.google.com/github/Ivan35-arch/movie_reccomender-system/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
!git clone https://github.com/Ivan35-arch/movie_reccomender-system.git

fatal: destination path 'movie_reccomender-system' already exists and is not an empty directory.


In [32]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

print('All imports successful')

All imports successful


In [33]:
import pandas as pd

# Files uploaded directly to the root session directory
ratings = pd.read_csv("ratings.csv")
movies  = pd.read_csv("movies.csv")
links   = pd.read_csv("links.csv")

print(movies.head())

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  


In [34]:
print('=== RATINGS ===')
print(f'Total ratings    : {len(ratings):,}')
print(f'Unique users     : {ratings["userId"].nunique():,}')
print(f'Unique movies    : {ratings["movieId"].nunique():,}')
print(f'Rating range     : {ratings["rating"].min()} - {ratings["rating"].max()}')
print(f'Average rating   : {ratings["rating"].mean():.2f}')
print()
print('=== RATING DISTRIBUTION ===')
print(ratings['rating'].value_counts().sort_index())

=== RATINGS ===
Total ratings    : 100,836
Unique users     : 610
Unique movies    : 9,724
Rating range     : 0.5 - 5.0
Average rating   : 3.50

=== RATING DISTRIBUTION ===
rating
0.5     1370
1.0     2811
1.5     1791
2.0     7551
2.5     5550
3.0    20047
3.5    13136
4.0    26818
4.5     8551
5.0    13211
Name: count, dtype: int64


In [35]:
# Merge ratings with movie titles for exploration
data = pd.merge(ratings, movies, on='movieId')
print('Merged dataset shape:', data.shape)
print()
data.head()

Merged dataset shape: (100836, 6)



,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [36]:
# Most rated movies
print('Top 10 most rated movies:')
data.groupby('title')['rating'].count().sort_values(ascending=False).head(10)

Top 10 most rated movies:


,rating
title,
Forrest Gump (1994),329
"Shawshank Redemption, The (1994)",317
Pulp Fiction (1994),307
"Silence of the Lambs, The (1991)",279
"Matrix, The (1999)",278
Star Wars: Episode IV - A New Hope (1977),251
Jurassic Park (1993),238
Braveheart (1995),237
Terminator 2: Judgment Day (1991),224


In [37]:
# Highest rated movies (minimum 100 ratings)
print('Top 10 highest rated movies (min 100 ratings):')
movie_stats = data.groupby('title')['rating'].agg(['mean', 'count'])
movie_stats[movie_stats['count'] >= 100].sort_values('mean', ascending=False).head(10)

Top 10 highest rated movies (min 100 ratings):


,mean,count
title,,
"Shawshank Redemption, The (1994)",4.429022,317
"Godfather, The (1972)",4.289062,192
Fight Club (1999),4.272936,218
"Godfather: Part II, The (1974)",4.259690,129
"Departed, The (2006)",4.252336,107
Goodfellas (1990),4.250000,126
Casablanca (1942),4.240000,100
"Dark Knight, The (2008)",4.238255,149
"Usual Suspects, The (1995)",4.237745,204


In [38]:
# Filter to reduce sparsity:
# Keep users who rated at least 20 movies
# Keep movies that have at least 20 ratings

MIN_USER_RATINGS  = 20
MIN_MOVIE_RATINGS = 20

user_counts  = ratings['userId'].value_counts()
movie_counts = ratings['movieId'].value_counts()

active_users  = user_counts[user_counts  >= MIN_USER_RATINGS].index
active_movies = movie_counts[movie_counts >= MIN_MOVIE_RATINGS].index

filtered = ratings[
    ratings['userId'].isin(active_users) &
    ratings['movieId'].isin(active_movies)
]

print(f'Original ratings : {len(ratings):,}')
print(f'Filtered ratings : {len(filtered):,}')
print(f'Active users     : {filtered["userId"].nunique():,}')
print(f'Active movies    : {filtered["movieId"].nunique():,}')

Original ratings : 100,836
Filtered ratings : 67,898
Active users     : 610
Active movies    : 1,297


In [39]:
# Build user-item matrix
# Rows = users, Columns = movies, Values = ratings (0 if not rated)

user_item_matrix = filtered.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
).fillna(0)

print(f'User-item matrix shape: {user_item_matrix.shape}')
print(f'Matrix density: {(user_item_matrix > 0).sum().sum() / (user_item_matrix.shape[0] * user_item_matrix.shape[1]):.2%}')
user_item_matrix.head()

User-item matrix shape: (610, 1297)
Matrix density: 8.58%


movieId,1,2,3,5,6,7,10,11,16,17,...,122920,122922,134130,134853,139385,148626,152081,164179,166528,168252
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,0.0,4.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [40]:
# Compute user-user cosine similarity
# Each row is a user vector in movie-rating space
# Cosine similarity measures the angle between user vectors
# Score of 1.0 = identical taste, 0.0 = no overlap, -1.0 = opposite taste

print('Computing cosine similarity matrix...')
user_similarity = cosine_similarity(user_item_matrix)

# Wrap in DataFrame for easy lookup
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

print(f'Similarity matrix shape: {user_similarity_df.shape}')
print()
print('Sample similarity scores (first 5 users):')
user_similarity_df.iloc[:5, :5]

Computing cosine similarity matrix...
Similarity matrix shape: (610, 610)

Sample similarity scores (first 5 users):


userId,1,2,3,4,5
userId,,,,,
1,1.000000,0.034691,0.120593,0.271535,0.151024
2,0.034691,1.000000,0.000000,0.005255,0.019629
3,0.120593,0.000000,1.000000,0.005205,0.009720
4,0.271535,0.005255,0.005205,1.000000,0.148323
5,0.151024,0.019629,0.009720,0.148323,1.000000


In [41]:
# Inspect most similar users to user #1 as a sanity check
sample_user = user_item_matrix.index[0]
similar_users = user_similarity_df[sample_user].sort_values(ascending=False)

print(f'Top 10 most similar users to user {sample_user}:')
print(similar_users.head(11))  # head(11) to skip self (score=1.0)

Top 10 most similar users to user 1:
userId
1      1.000000
368    0.447079
91     0.433369
599    0.432526
313    0.431679
217    0.423776
469    0.421438
57     0.420379
288    0.419436
266    0.414428
19     0.409864
Name: 1, dtype: float64


In [42]:
def get_recommendations(user_id, user_item_matrix, user_similarity_df, n=10):
    """
    Generate movie recommendations for a user using
    user-user collaborative filtering.

    Steps:
    1. Get similarity scores for this user vs all others
    2. Compute weighted sum of all users' ratings
    3. Exclude movies the user has already rated
    4. Return top N movie IDs by predicted score
    """

    if user_id not in user_item_matrix.index:
        print(f'User {user_id} not in matrix — cold start fallback')
        # Cold start: return most popular movies
        popularity = (user_item_matrix > 0).sum(axis=0)
        return popularity.nlargest(n).index.tolist()

    # Similarity scores for this user
    sim_scores = user_similarity_df[user_id].values  # shape: (n_users,)

    # Weighted average of all users' ratings
    weighted_ratings = np.dot(sim_scores, user_item_matrix.values)

    # Normalize by sum of similarities
    sim_sum = np.abs(sim_scores).sum()
    if sim_sum > 0:
        weighted_ratings = weighted_ratings / sim_sum

    # Mask already-rated movies
    user_ratings = user_item_matrix.loc[user_id].values
    weighted_ratings[user_ratings > 0] = -1

    # Get top N
    top_indices = np.argsort(weighted_ratings)[::-1][:n]
    top_movie_ids = user_item_matrix.columns[top_indices].tolist()

    return top_movie_ids


# Test the function
test_user = user_item_matrix.index[0]
recs = get_recommendations(test_user, user_item_matrix, user_similarity_df, n=10)

print(f'Top 10 recommendations for user {test_user}:')
rec_titles = movies[movies['movieId'].isin(recs)][['movieId', 'title']]
print(rec_titles.to_string(index=False))

Top 10 recommendations for user 1:
 movieId                                                     title
      32                 Twelve Monkeys (a.k.a. 12 Monkeys) (1995)
     150                                          Apollo 13 (1995)
     318                          Shawshank Redemption, The (1994)
     588                                            Aladdin (1992)
     589                         Terminator 2: Judgment Day (1991)
     858                                     Godfather, The (1972)
    2762                                   Sixth Sense, The (1999)
    4993 Lord of the Rings: The Fellowship of the Ring, The (2001)
    5952             Lord of the Rings: The Two Towers, The (2002)
    7153     Lord of the Rings: The Return of the King, The (2003)


In [43]:
# Train/test split — hold out 20% of ratings for evaluation
train_data, test_data = train_test_split(filtered, test_size=0.2, random_state=42)

print(f'Train size : {len(train_data):,}')
print(f'Test size  : {len(test_data):,}')

Train size : 54,318
Test size  : 13,580


In [44]:
# Build train matrix and similarity for evaluation
train_matrix = train_data.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
).fillna(0)

train_similarity = cosine_similarity(train_matrix)
train_similarity_df = pd.DataFrame(
    train_similarity,
    index=train_matrix.index,
    columns=train_matrix.index
)

print('Train matrix built:', train_matrix.shape)

Train matrix built: (610, 1297)


In [45]:
# Evaluate using RMSE on test set
actuals    = []
predicted  = []
eval_limit = 500  # sample for speed

test_sample = test_data[
    test_data['userId'].isin(train_matrix.index) &
    test_data['movieId'].isin(train_matrix.columns)
].head(eval_limit)

for _, row in test_sample.iterrows():
    uid = row['userId']
    mid = row['movieId']
    actual = row['rating']

    sim_scores   = train_similarity_df[uid].values
    movie_idx    = list(train_matrix.columns).index(mid)
    movie_ratings = train_matrix.iloc[:, movie_idx].values

    sim_sum = np.abs(sim_scores).sum()
    pred = np.dot(sim_scores, movie_ratings) / sim_sum if sim_sum > 0 else 3.0
    pred = np.clip(pred, 0.5, 5.0)

    actuals.append(actual)
    predicted.append(pred)

rmse = np.sqrt(mean_squared_error(actuals, predicted))
mae  = np.mean(np.abs(np.array(actuals) - np.array(predicted)))

print(f'Evaluation on {len(actuals)} test samples:')
print(f'RMSE : {rmse:.4f}')
print(f'MAE  : {mae:.4f}')
print()
print('RMSE < 1.0 is considered good for collaborative filtering')

Evaluation on 500 test samples:
RMSE : 3.0627
MAE  : 2.8913

RMSE < 1.0 is considered good for collaborative filtering


In [46]:
# Precision@K — what fraction of top K recs are actually relevant
# A movie is 'relevant' if the user rated it >= 4.0

def precision_at_k(user_id, k=10, relevance_threshold=4.0):
    # Get actual highly rated movies from test set
    user_test = test_data[
        (test_data['userId'] == user_id) &
        (test_data['rating'] >= relevance_threshold)
    ]['movieId'].tolist()

    if not user_test:
        return None

    recs = get_recommendations(user_id, train_matrix, train_similarity_df, n=k)
    hits = len(set(recs) & set(user_test))
    return hits / k


# Evaluate precision@10 on sample of users
sample_users = test_data['userId'].unique()[:100]
precisions = [precision_at_k(u) for u in sample_users if u in train_matrix.index]
precisions = [p for p in precisions if p is not None]

print(f'Precision@10 on {len(precisions)} users: {np.mean(precisions):.4f}')
print(f'Meaning: {np.mean(precisions)*100:.1f}% of top 10 recommendations are relevant')

Precision@10 on 100 users: 0.2870
Meaning: 28.7% of top 10 recommendations are relevant


In [47]:
# Train final model on ALL data (not just train split)
print('Training final model on full dataset...')

final_user_item_matrix = filtered.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
).fillna(0)

final_user_similarity = cosine_similarity(final_user_item_matrix)

print(f'Final matrix shape     : {final_user_item_matrix.shape}')
print(f'Final similarity shape : {final_user_similarity.shape}')

Training final model on full dataset...
Final matrix shape     : (610, 1297)
Final similarity shape : (610, 610)


In [48]:
# Save model
os.makedirs('model', exist_ok=True)

model_data = {
    'user_similarity':  final_user_similarity,
    'user_item_matrix': final_user_item_matrix
}

joblib.dump(model_data, 'model/movie_recommender_model.joblib', compress=3)

file_size = os.path.getsize('model/movie_recommender_model.joblib') / (1024 * 1024)
print(f'Model saved to model/movie_recommender_model.joblib')
print(f'File size: {file_size:.1f} MB')

Model saved to model/movie_recommender_model.joblib
File size: 2.6 MB


In [49]:
# Reload and verify it works correctly
loaded = joblib.load('model/movie_recommender_model.joblib')

loaded_similarity  = loaded['user_similarity']
loaded_matrix      = loaded['user_item_matrix']

print('Model loaded successfully')
print(f'user_similarity shape  : {loaded_similarity.shape}')
print(f'user_item_matrix shape : {loaded_matrix.shape}')
print(f'Users in model         : {len(loaded_matrix.index):,}')
print(f'Movies in model        : {len(loaded_matrix.columns):,}')

Model loaded successfully
user_similarity shape  : (610, 610)
user_item_matrix shape : (610, 1297)
Users in model         : 610
Movies in model        : 1,297


In [50]:
# Final end-to-end test using loaded model
loaded_similarity_df = pd.DataFrame(
    loaded_similarity,
    index=loaded_matrix.index,
    columns=loaded_matrix.index
)

test_user = loaded_matrix.index[0]
final_recs = get_recommendations(test_user, loaded_matrix, loaded_similarity_df, n=10)

print(f'Final recommendations for user {test_user}:')
rec_titles = movies[movies['movieId'].isin(final_recs)][['movieId', 'title', 'genres']]
print(rec_titles.to_string(index=False))

Final recommendations for user 1:
 movieId                                                     title                                      genres
      32                 Twelve Monkeys (a.k.a. 12 Monkeys) (1995)                     Mystery|Sci-Fi|Thriller
     150                                          Apollo 13 (1995)                        Adventure|Drama|IMAX
     318                          Shawshank Redemption, The (1994)                                 Crime|Drama
     588                                            Aladdin (1992) Adventure|Animation|Children|Comedy|Musical
     589                         Terminator 2: Judgment Day (1991)                               Action|Sci-Fi
     858                                     Godfather, The (1972)                                 Crime|Drama
    2762                                   Sixth Sense, The (1999)                        Drama|Horror|Mystery
    4993 Lord of the Rings: The Fellowship of the Ring, The (2001)            

In [51]:
print('=== MODEL SUMMARY ===')
print(f'Algorithm       : User-User Collaborative Filtering')
print(f'Similarity      : Cosine Similarity')
print(f'Users           : {loaded_matrix.shape[0]:,}')
print(f'Movies          : {loaded_matrix.shape[1]:,}')
print(f'RMSE            : {rmse:.4f}')
print(f'MAE             : {mae:.4f}')
print(f'Precision@10    : {np.mean(precisions):.4f}')
print(f'Saved to        : model/movie_recommender_model.joblib')
print()
print('Ready to use in Flask API via:')
print("  model_data = joblib.load('model/movie_recommender_model.joblib')")
print("  user_similarity  = model_data['user_similarity']")
print("  user_item_matrix = model_data['user_item_matrix']")

=== MODEL SUMMARY ===
Algorithm       : User-User Collaborative Filtering
Similarity      : Cosine Similarity
Users           : 610
Movies          : 1,297
RMSE            : 3.0627
MAE             : 2.8913
Precision@10    : 0.2870
Saved to        : model/movie_recommender_model.joblib

Ready to use in Flask API via:
  model_data = joblib.load('model/movie_recommender_model.joblib')
  user_similarity  = model_data['user_similarity']
  user_item_matrix = model_data['user_item_matrix']
